# AI-Powered Interview Question Generator

Generates custom technical and behavioral interview questions for interns,
tailored to a role/skills/experience profile, using an LLM via OpenRouter.
New questions are deduplicated against a seed "existing question bank".

In [1]:
import os
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

from interview_gen.question_bank import load_seed_question_bank
from interview_gen.generator import generate_questions

assert os.environ.get("OPENROUTER_API_KEY"), (
    "OPENROUTER_API_KEY not found. Create a .env file in this directory "
    "with OPENROUTER_API_KEY=<your key> (see .env.example)."
)
print("API key loaded.")

API key loaded.


## 1. Seed question bank

Represents the "existing question bank" data input from the brief, and
doubles as the deduplication reference for newly generated questions.

In [2]:
question_bank = load_seed_question_bank()
print(f"Seed question bank: {len(question_bank)} questions")
question_bank

Seed question bank: 18 questions


,question,category,topic,difficulty
0,What is the difference between a list and a tu...,technical,Python,beginner
1,How would you handle missing values in a dataset?,technical,Data Analysis,beginner
2,Explain the difference between INNER JOIN and ...,technical,SQL,intermediate
3,"What is overfitting, and how can it be prevented?",technical,Machine Learning,intermediate
4,Describe how you would design a REST API for a...,technical,Product,intermediate
5,"What is the time complexity of binary search, ...",technical,Python,intermediate
6,How do you approach debugging a failing test i...,technical,Python,beginner
7,What is the difference between supervised and ...,technical,Machine Learning,beginner
8,How would you optimize a slow SQL query?,technical,SQL,advanced
9,"What version control workflow have you used, a...",technical,Project Management,beginner


## 2. Sample intern profiles

Structured profiles standing in for a job description + intern background.

In [3]:
sample_profiles = [
    {
        "role": "Data Analyst Intern",
        "tech_skills": ["Python", "SQL"],
        "experience_level": "beginner",
        "focus_topics": ["data cleaning", "visualization"],
    },
    {
        "role": "Backend Engineering Intern",
        "tech_skills": ["Python", "REST APIs", "PostgreSQL"],
        "experience_level": "intermediate",
        "focus_topics": ["API design", "database performance"],
    },
    {
        "role": "Machine Learning Intern",
        "tech_skills": ["Python", "scikit-learn", "pandas"],
        "experience_level": "intermediate",
        "focus_topics": ["model evaluation", "feature engineering"],
    },
]
sample_profiles

[{'role': 'Data Analyst Intern',
  'tech_skills': ['Python', 'SQL'],
  'experience_level': 'beginner',
  'focus_topics': ['data cleaning', 'visualization']},
 {'role': 'Backend Engineering Intern',
  'tech_skills': ['Python', 'REST APIs', 'PostgreSQL'],
  'experience_level': 'intermediate',
  'focus_topics': ['API design', 'database performance']},
 {'role': 'Machine Learning Intern',
  'tech_skills': ['Python', 'scikit-learn', 'pandas'],
  'experience_level': 'intermediate',
  'focus_topics': ['model evaluation', 'feature engineering']}]

## 3. Generate questions for the first profile

In [4]:
profile = sample_profiles[0]
result_df = generate_questions(profile, question_bank)
print(f"Generated {len(result_df)} new (non-duplicate) questions for: {profile['role']}")
result_df

Generated 6 new (non-duplicate) questions for: Data Analyst Intern


,question,category,difficulty
0,Can you explain how you would handle missing v...,technical,beginner
1,Write an SQL query to filter and aggregate dat...,technical,beginner
2,How would you create a basic visualization (li...,technical,beginner
3,Describe a time when you encountered inconsist...,behavioral,beginner
4,How do you prioritize tasks when working on da...,behavioral,beginner
5,If you were given a dataset with date fields f...,technical,beginner


## 4. Grow the bank and generate for the remaining profiles

Each new profile's generated questions are deduplicated against the
seed bank *and* every question generated so far in this run.

In [5]:
growing_bank = pd.concat(
    [question_bank[["question"]], result_df[["question"]]], ignore_index=True
)

all_results = {profile["role"]: result_df}

for profile in sample_profiles[1:]:
    new_questions = generate_questions(profile, growing_bank)
    print(f"Generated {len(new_questions)} new questions for: {profile['role']}")
    all_results[profile["role"]] = new_questions
    growing_bank = pd.concat(
        [growing_bank, new_questions[["question"]]], ignore_index=True
    )

for role, df in all_results.items():
    print(f"\n=== {role} ===")
    display(df)

Generated 6 new questions for: Backend Engineering Intern


Generated 6 new questions for: Machine Learning Intern

=== Data Analyst Intern ===


,question,category,difficulty
0,Can you explain how you would handle missing v...,technical,beginner
1,Write an SQL query to filter and aggregate dat...,technical,beginner
2,How would you create a basic visualization (li...,technical,beginner
3,Describe a time when you encountered inconsist...,behavioral,beginner
4,How do you prioritize tasks when working on da...,behavioral,beginner
5,If you were given a dataset with date fields f...,technical,beginner



=== Backend Engineering Intern ===


,question,category,difficulty
0,How would you design a RESTful API to handle h...,technical,intermediate
1,Can you explain how you would optimize a slow-...,technical,intermediate
2,Describe a time you encountered a database per...,behavioral,intermediate
3,How do you approach designing an API that bala...,technical,intermediate
4,Tell me about a project where you implemented ...,technical,intermediate
5,Describe a situation where you had to collabor...,behavioral,intermediate



=== Machine Learning Intern ===


,question,category,difficulty
0,Can you explain the differences between precis...,technical,intermediate
1,How would you approach feature engineering for...,technical,intermediate
2,Describe a time when you had to troubleshoot a...,behavioral,intermediate
3,What are the key assumptions behind k-fold cro...,technical,intermediate
4,Can you walk through a recent project where yo...,technical,intermediate
5,Describe a situation where you had to collabor...,behavioral,intermediate


## Reusable helper

`generate_questions(profile, question_bank)` can be called with any new
profile dict (`role`, `tech_skills`, `experience_level`, `focus_topics`) and
the current question bank DataFrame to produce a fresh, deduplicated
question set for that role.